# Chirurgie ONNX : Normalisation pré-Concat pour YOLOP


**Objectif** : Diviser les coordonnées (canaux 0-4) par 640 pour les ramener entre 0 et 1, tout en préservant les probabilités (canaux 4-6).

In [1]:
!pip install --upgrade onnx onnxruntime onnx-graphsurgeon

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 117.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 117.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 552.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 119.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 152.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.24.4
    Uninstalling protobuf-4.24.4:
      Successfully uninstalled protobuf-4.24.4
  Attempting uninstall: onnx
    Found existing installation: onnx 1.15.0rc2
    Uninstalling onnx-1.15.0rc2:
      Successfully uninstalled onnx-1.15.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 24

In [2]:

import os
import onnx
from onnx import helper, numpy_helper, TensorProto
import numpy as np
from collections import defaultdict, deque


In [3]:

BASE_DIR = os.getcwd()
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640.onnx")
OUTPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640-normalized.onnx")

TARGET_CONCAT_NODE = "Concat_1534"
INPUT_BRANCHES = [
    {"tensor": "1498", "reshape_node": "Reshape_917", "source": "ScatterND_914"},
    {"tensor": "1828", "reshape_node": "Reshape_1225", "source": "ScatterND_1222"},
    {"tensor": "2158", "reshape_node": "Reshape_1533", "source": "ScatterND_1530"}
]

NORMALIZATION_FACTOR = 640.0

print(f" Modèle d'entrée  : {INPUT_MODEL_PATH}")
print(f" Modèle de sortie : {OUTPUT_MODEL_PATH}")


 Modèle d'entrée  : /workspace/weights/yolop-640-640.onnx
 Modèle de sortie : /workspace/weights/yolop-640-640-normalized.onnx


In [4]:

def find_node_by_name(model, node_name):
  
    for node in model.graph.node:
        if node.name == node_name:
            return node
    return None


In [5]:

def find_node_producing_tensor(model, tensor_name):
    
    for node in model.graph.node:
        if tensor_name in node.output:
            return node
    return None


In [6]:

def create_constant_tensor(name, value, dtype=TensorProto.FLOAT):
 
    if isinstance(value, (int, float)):
        value = np.array([value], dtype=np.float32 if dtype == TensorProto.FLOAT else np.int64)
    elif not isinstance(value, np.ndarray):
        value = np.array(value)
    return helper.make_tensor(name, dtype, value.shape, value.flatten().tolist())


In [7]:

def topological_sort_nodes(graph):
   
    print("    Tri topologique manuel en cours...")
    
    tensor_to_producer = {}
    for node in graph.node:
        for output in node.output:
            tensor_to_producer[output] = node
    
    available_tensors = set()
    for input_tensor in graph.input:
        available_tensors.add(input_tensor.name)
    for init in graph.initializer:
        available_tensors.add(init.name)
    
    dependencies = defaultdict(set)  
    
    for node in graph.node:
        for input_tensor in node.input:
            if input_tensor in tensor_to_producer:
                producer_node = tensor_to_producer[input_tensor]
                dependencies[node.name].add(producer_node.name)
    
    sorted_nodes = []
    nodes_dict = {node.name: node for node in graph.node}
    in_degree = {node.name: len(dependencies[node.name]) for node in graph.node}
    
    queue = deque([node.name for node in graph.node if in_degree[node.name] == 0])
    
    while queue:
        current_name = queue.popleft()
        current_node = nodes_dict[current_name]
        sorted_nodes.append(current_node)
        
        for output in current_node.output:
            available_tensors.add(output)
        
        for node_name in nodes_dict:
            if current_name in dependencies[node_name]:
                dependencies[node_name].remove(current_name)
                in_degree[node_name] -= 1
                if in_degree[node_name] == 0:
                    queue.append(node_name)
    
    if len(sorted_nodes) != len(graph.node):
        remaining = len(graph.node) - len(sorted_nodes)
        print(f"    Cycle détecté ! {remaining} nœuds non triés")
        
        for node in graph.node:
            if node not in sorted_nodes:
                sorted_nodes.append(node)
    
    print(f"    {len(sorted_nodes)} nœuds triés")
    return sorted_nodes



In [8]:

def perform_surgical_modification(model_path, output_path):
   
    print("\n" + "="*70)
    print("   CHIRURGIE ONNX : NORMALISATION PRÉ-CONCAT")
    print("="*70)
    
    print("\n[0/5]  Chargement du modèle...")
    model = onnx.load(model_path)
    graph = model.graph
    
    concat_node = find_node_by_name(model, TARGET_CONCAT_NODE)
    if concat_node is None:
        print(f" ERREUR : Nœud '{TARGET_CONCAT_NODE}' introuvable")
        return
    
    print(f" Nœud cible '{TARGET_CONCAT_NODE}' trouvé")
    print(f"   Entrées actuelles : {list(concat_node.input)}")
    
    print("\n[1/5]  Repérage des zones d'intervention...")
    
    branch_info = []
    for i, branch in enumerate(INPUT_BRANCHES):
        tensor_name = branch["tensor"]
        reshape_name = branch["reshape_node"]
        source_name = branch.get("source")
        
        if tensor_name not in concat_node.input:
            print(f"  Branche {i+1} : Tenseur '{tensor_name}' absent des entrées du Concat")
            continue
        
        reshape_node = find_node_by_name(model, reshape_name)
        if reshape_node is None:
            print(f"  Branche {i+1} : Reshape '{reshape_name}' introuvable")
            continue
        
        source_tensor = reshape_node.input[0]
        
        branch_info.append({
            "index": i,
            "tensor": tensor_name,
            "reshape_node": reshape_name,
            "reshape": reshape_node,
            "source_tensor": source_tensor,
            "position": list(concat_node.input).index(tensor_name)
        })
        
        print(f" Branche {i+1} : '{tensor_name}' <- Reshape('{source_tensor}') - Position {branch_info[-1]['position']}")
    
    if len(branch_info) == 0:
        print(" ERREUR : Aucune branche valide trouvée")
        return
    
    print("\n[2/5]  Préparation de la modification...")
    
    new_nodes = []
    new_initializers = []
    
    print("\n[3/5]   Installation des blocs régulateurs...")
    
    for branch in branch_info:
        branch_idx = branch["index"]
        tensor_name = branch["tensor"]
        source_tensor = branch["source_tensor"]
        position = branch["position"]
        
        print(f"\n    Traitement branche {branch_idx + 1} ('{tensor_name}')...")
        print(f"      Source : {source_tensor}")
        
        print(f"      [3.1] Reshape vers 3D [batch, -1, 6]...")
        
        reshape_3d_output = f"{source_tensor}_reshaped_3d"
        reshape_3d_shape_name = f"{source_tensor}_shape_3d"
        
        reshape_3d_shape = np.array([0, -1, 6], dtype=np.int64)
        reshape_3d_shape_tensor = create_constant_tensor(
            reshape_3d_shape_name,
            reshape_3d_shape,
            TensorProto.INT64
        )
        new_initializers.append(reshape_3d_shape_tensor)
        
        reshape_3d_node = helper.make_node(
            "Reshape",
            inputs=[source_tensor, reshape_3d_shape_name],
            outputs=[reshape_3d_output],
            name=f"Reshape_{source_tensor}_to_3d_normalize"
        )
        new_nodes.append(reshape_3d_node)
        print(f"         ✓ Reshape 3D : {reshape_3d_output}")
        
        print(f"      [3.2] Slice des coordonnées (canaux 0-3)...")
        
        slice_coords_output = f"{source_tensor}_coords"
        slice_coords_starts = f"{source_tensor}_coords_starts"
        slice_coords_ends = f"{source_tensor}_coords_ends"
        slice_coords_axes = f"{source_tensor}_coords_axes"
        
        new_initializers.append(create_constant_tensor(slice_coords_starts, np.array([0], dtype=np.int64), TensorProto.INT64))
        new_initializers.append(create_constant_tensor(slice_coords_ends, np.array([4], dtype=np.int64), TensorProto.INT64))
        new_initializers.append(create_constant_tensor(slice_coords_axes, np.array([2], dtype=np.int64), TensorProto.INT64))
        
        slice_coords_node = helper.make_node(
            "Slice",
            inputs=[reshape_3d_output, slice_coords_starts, slice_coords_ends, slice_coords_axes],
            outputs=[slice_coords_output],
            name=f"Slice_{source_tensor}_coords"
        )
        new_nodes.append(slice_coords_node)
        print(f"         ✓ Slice coords : {slice_coords_output}")
        
        print(f"      [3.3] Slice des probabilités (canaux 4-5)...")
        
        slice_probs_output = f"{source_tensor}_probs"
        slice_probs_starts = f"{source_tensor}_probs_starts"
        slice_probs_ends = f"{source_tensor}_probs_ends"
        slice_probs_axes = f"{source_tensor}_probs_axes"
        
        new_initializers.append(create_constant_tensor(slice_probs_starts, np.array([4], dtype=np.int64), TensorProto.INT64))
        new_initializers.append(create_constant_tensor(slice_probs_ends, np.array([6], dtype=np.int64), TensorProto.INT64))
        new_initializers.append(create_constant_tensor(slice_probs_axes, np.array([2], dtype=np.int64), TensorProto.INT64))
        
        slice_probs_node = helper.make_node(
            "Slice",
            inputs=[reshape_3d_output, slice_probs_starts, slice_probs_ends, slice_probs_axes],
            outputs=[slice_probs_output],
            name=f"Slice_{source_tensor}_probs"
        )
        new_nodes.append(slice_probs_node)
        print(f"         ✓ Slice probs : {slice_probs_output}")
        
        print(f"      [3.4] Division des coordonnées par {NORMALIZATION_FACTOR}...")
        
        div_output = f"{source_tensor}_coords_normalized"
        div_constant_name = f"{source_tensor}_div_constant"
        
        div_constant = create_constant_tensor(div_constant_name, NORMALIZATION_FACTOR, TensorProto.FLOAT)
        new_initializers.append(div_constant)
        
        div_node = helper.make_node(
            "Div",
            inputs=[slice_coords_output, div_constant_name],
            outputs=[div_output],
            name=f"Div_{source_tensor}_normalize"
        )
        new_nodes.append(div_node)
        print(f"         ✓ Div : {div_output}")
        
        print(f"      [3.5] Concat des parties normalisées...")
        
        concat_output = f"{source_tensor}_recombined"
        
        concat_node_inter = helper.make_node(
            "Concat",
            inputs=[div_output, slice_probs_output],
            outputs=[concat_output],
            name=f"Concat_{source_tensor}_recombine",
            axis=2 
        )
        new_nodes.append(concat_node_inter)
        print(f"         ✓ Concat : {concat_output}")
        
        print(f"      [3.6] Reshape vers forme aplatie [batch, -1]...")
        
        reshape_back_output = f"{source_tensor}_normalized_flat"
        reshape_back_shape_name = f"{source_tensor}_shape_flat"
        
        reshape_back_shape = np.array([0, -1], dtype=np.int64)
        reshape_back_shape_tensor = create_constant_tensor(
            reshape_back_shape_name,
            reshape_back_shape,
            TensorProto.INT64
        )
        new_initializers.append(reshape_back_shape_tensor)
        
        reshape_back_node = helper.make_node(
            "Reshape",
            inputs=[concat_output, reshape_back_shape_name],
            outputs=[reshape_back_output],
            name=f"Reshape_{source_tensor}_flatten_normalized"
        )
        new_nodes.append(reshape_back_node)
        print(f"         ✓ Reshape flat : {reshape_back_output}")
        
        print(f"      [3.7] Redirection du Reshape original...")
        
        original_reshape = branch["reshape"]
        
        original_reshape.input[0] = reshape_back_output
        
        print(f"         ✓ Reshape '{original_reshape.name}' redirigé vers {reshape_back_output}")
        print(f"       Branche {branch_idx + 1} : Pipeline installé")
    
    print(f"\n[4/5] Vérification...")
    processed_count = len(branch_info)
    print(f"    {processed_count} branches traitées")
    
    print("\n[5/5]  Intégration dans le graphe avec tri topologique...")
    
    
    for initializer in new_initializers:
        graph.initializer.append(initializer)
    
    print(f"   {len(new_initializers)} nouvelles constantes ajoutées")
    
    for node in new_nodes:
        graph.node.append(node)
    
    print(f"    {len(new_nodes)} nouveaux nœuds ajoutés temporairement")
    
    print("\n Tri topologique complet du graphe...")
    
    sorted_nodes = topological_sort_nodes(graph)
    
    del graph.node[:]
    graph.node.extend(sorted_nodes)
    
    print(" Graphe reconstruit avec ordre topologique")
    
    print("\n Sauvegarde du modèle modifié...")
    
    try:
        onnx.checker.check_model(model)
        print(" Validation ONNX réussie")
    except Exception as e:
        print(f"  Avertissement lors de la validation : {str(e)}")
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    onnx.save(model, output_path)
    
    print("\n" + "="*70)
    print("   CHIRURGIE TERMINÉE AVEC SUCCÈS")
    print("="*70)
    
    if os.path.exists(output_path):
        size_original = os.path.getsize(model_path) / (1024 * 1024)
        size_modified = os.path.getsize(output_path) / (1024 * 1024)
        
        print(f"\n Statistiques :")
        print(f"   Original  : {size_original:>8.2f} MB")
        print(f"   Modifié   : {size_modified:>8.2f} MB")
        print(f"   Différence: {size_modified - size_original:>+8.2f} MB")
        print(f"\n Modèle modifié :")
        print(f"   {output_path}")
        print(f"\n Modifications appliquées :")
        print(f"   • {processed_count} branches normalisées")
        print(f"   • Pipeline : Source → Reshape3D → Slice → Div → Concat → ReshapeFlat → Reshape_original")
        print(f"   • Coordonnées (canaux 0-3) divisées par {NORMALIZATION_FACTOR}")
        print(f"   • Probabilités (canaux 4-5) préservées")
        print(f"   • Forme finale identique au modèle original")
        print(f"   • Les Reshape originaux (917, 1225, 1533) conservés et redirigés")
        print(f"   • Graphe trié topologiquement (algorithme de Kahn)")
        print("\n" + "="*70)
    else:
        print("\n ERREUR : Le fichier de sortie n'a pas été créé")



In [9]:

if __name__ == "__main__":
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f" ERREUR : Modèle introuvable : {INPUT_MODEL_PATH}")
        print(f"\nVeuillez placer votre modèle YOLOP dans le dossier 'weights/'")
    else:
        
        perform_surgical_modification(INPUT_MODEL_PATH, OUTPUT_MODEL_PATH)


   CHIRURGIE ONNX : NORMALISATION PRÉ-CONCAT

[0/5]  Chargement du modèle...
 Nœud cible 'Concat_1534' trouvé
   Entrées actuelles : ['1498', '1828', '2158']

[1/5]  Repérage des zones d'intervention...
 Branche 1 : '1498' <- Reshape('1491') - Position 0
 Branche 2 : '1828' <- Reshape('1821') - Position 1
 Branche 3 : '2158' <- Reshape('2151') - Position 2

[2/5]  Préparation de la modification...

[3/5]   Installation des blocs régulateurs...

    Traitement branche 1 ('1498')...
      Source : 1491
      [3.1] Reshape vers 3D [batch, -1, 6]...
         ✓ Reshape 3D : 1491_reshaped_3d
      [3.2] Slice des coordonnées (canaux 0-3)...
         ✓ Slice coords : 1491_coords
      [3.3] Slice des probabilités (canaux 4-5)...
         ✓ Slice probs : 1491_probs
      [3.4] Division des coordonnées par 640.0...
         ✓ Div : 1491_coords_normalized
      [3.5] Concat des parties normalisées...
         ✓ Concat : 1491_recombined
      [3.6] Reshape vers forme aplatie [batch, -1]...
     